## 쿠텐 top10 상품 추출 및 모든 리뷰데이터 추출 코드

In [2]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import requests
import time
import random
import json
import re
from datetime import datetime

# ── 설정 ──────────────────────────────────────────────────────────
RANK_SAVE_FILE   = "qoo10_rankings_current.jsonl"  
REVIEW_SAVE_FILE = "qoo10_reviews_master.jsonl"    

# ── 함수: 단일 상품 리뷰 전체 수집 (사용자 제공 로직 유지) ──────────────
def get_qoo10_reviews_all(gd_no, product_name):
    all_reviews = []
    page     = 1
    base_url = "https://www.qoo10.jp/gmkt.inc/Goods/GoodsReviewAjaxAppend.aspx"

    headers = {
        "User-Agent"     : "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
        "Accept"         : "text/html, */*",
        "Accept-Language": "ja,en-US;q=0.9,en;q=0.8,ko;q=0.7",
        "Referer"        : f"https://www.qoo10.jp/item/x/{gd_no}",
        "X-Requested-With": "XMLHttpRequest",
    }

    print(f"\n   🚀 [{product_name[:30]}...] 리뷰 수집 시작...")

    while True:
        params = {
            "gd_no"             : gd_no,
            "group_code"        : "2",
            "page_no"           : page,
            "page_size"         : "100",
            "sort_type"         : "P",
            "contents_cnt"      : "0",
            "___cache_expire___": int(time.time() * 1000),
        }
        try:
            response = requests.get(base_url, params=params, headers=headers, timeout=15)
            if response.status_code != 200: break

            html_content = response.text.strip()
            if not html_content: break

            soup = BeautifulSoup(html_content, "html.parser")
            review_items = soup.find_all("li", recursive=False) or soup.select("li")
            if not review_items: break

            for item in review_items:
                txt_tag = item.select_one(".review_txt")
                if not txt_tag: continue

                content   = txt_tag.get_text(strip=True)
                score_tag = item.select_one(".score")
                rating    = score_tag.get_text(strip=True) if score_tag else "5"
                user_info = item.select_one(".review_user_info")
                user_meta = user_info.get_text(" | ", strip=True) if user_info else ""
                type_tag  = item.select_one(".review_user_type")
                skin_info = type_tag.get_text(strip=True) if type_tag else ""

                all_reviews.append({
                    "gd_no"   : gd_no,
                    "Page"    : page,
                    "Rating"  : rating,
                    "Review"  : content,
                    "UserInfo": user_meta,
                    "SkinType": skin_info,
                })

            print(f"   🔄 {page}페이지 수집 중... (누적: {len(all_reviews)}개)", end="\r")
            page += 1
            time.sleep(0.5)

        except Exception as e:
            print(f"\n   ❌ 리뷰 에러: {e}")
            break

    print(f"\n   ✨ 리뷰 완료: {len(all_reviews)}개")
    return all_reviews


# ── 메인 실행부 ──────────────────────────────────────────────────
options = uc.ChromeOptions()
options.add_argument('--headless') 
options.add_argument('--window-size=1920,1080')
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_argument('--incognito')
options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36')

driver = None
try:
    driver = uc.Chrome(options=options)
    # 카테고리 베이스 링크
    target_url = "https://www.qoo10.jp/cat/120000012"

    print(f"📡 큐텐 카테고리 접속 중: {target_url}")
    driver.get(target_url)
    time.sleep(random.uniform(6, 9))

    # 리스트 로딩을 위해 스크롤 내림
    driver.execute_script("window.scrollTo(0, 2000);")
    time.sleep(2)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    # li 구조의 상품 리스트 선택
    items = soup.select('ul#search_result_item_list > li')

    rank_data_list = []
    all_review_master = []
    rank_count = 1

    for item in items:
        if rank_count > 10: break

        try:
            # 1. PR 상품 제외 (ad_cps 클래스 또는 PR 배지 확인)
            is_pr = item.select_one(".ad_cps, .icon_pr, .item_pr") or "PR" in item.get_text()
            if is_pr: continue

            # 2. 리뷰 개수 확인 (리뷰 없는 상품 제외)
            review_tag = item.select_one(".review_total_count")
            reviews_raw = review_tag.get_text(strip=True) if review_tag else "0"
            reviews_num = int(re.sub(r'[^0-9]', '', reviews_raw)) if reviews_raw != "0" else 0
            
            if reviews_num == 0: continue # 리뷰 없으면 패스

            # 3. 상품 기본 정보 추출
            goods_code = item.get("goodscode")
            brand_tag  = item.select_one(".txt_brand")
            brand      = brand_tag.get_text(strip=True).replace("公式", "").strip() if brand_tag else "N/A"
            
            title_tag  = item.select_one("a.tt")
            title      = title_tag.get_text(strip=True) if title_tag else "N/A"
            
            price_tag  = item.select_one(".prc strong")
            price_str  = price_tag.get_text(strip=True) if price_tag else "0"

            # 평점 (별점 %를 5점 만점으로 변환)
            rating_star = item.select_one(".review_rating_star")
            rating = 0.0
            if rating_star and "style" in rating_star.attrs:
                width_match = re.search(r'width:\s*(\d+)%', rating_star["style"])
                if width_match:
                    rating = round(float(width_match.group(1)) / 20, 1)

            # 4. 요구하신 Rakuten 스타일 포맷팅
            product_info = {
                "rank": rank_count,
                "title": title,
                "shop_name": brand,
                "rating": rating,
                "reviews": reviews_num,
                "price": price_str,
                "trend": "Stay",
                "url": f"https://www.qoo10.jp/item/x/{goods_code}",
                "shop_id": "", # 큐텐 구조상 빈값 또는 브랜드번호 파싱 필요
                "item_id": goods_code,
                "platform": "Qoo10",
                "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            }

            print(f"\n📍 {rank_count}위 확정: [{brand}] {goods_code}")
            rank_data_list.append(product_info)

            # 5. 해당 상품의 모든 리뷰 수집 (AJAX)
            if goods_code:
                reviews_data = get_qoo10_reviews_all(goods_code, title)
                all_review_master.extend(reviews_data)
            
            rank_count += 1

        except Exception as e:
            continue

    # ── 저장 (JSONL 형식) ──
    with open(RANK_SAVE_FILE, "w", encoding="utf-8") as f:
        for entry in rank_data_list:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    with open(REVIEW_SAVE_FILE, "w", encoding="utf-8") as f:
        for review in all_review_master:
            f.write(json.dumps(review, ensure_ascii=False) + "\n")

    print(f"\n✨ 수집 완료: 상품 {len(rank_data_list)}개 / 총 리뷰 {len(all_review_master)}개")

except Exception as e:
    print(f"❌ 치명적 오류: {e}")
finally:
    if driver:
        driver.quit()

📡 큐텐 카테고리 접속 중: https://www.qoo10.jp/cat/120000012

📍 1위 확정: [DOPAMY] 1184035014

   🚀 [【公式】【1+1】日本初上陸！ Qoo10限定 ２つ自分で選...] 리뷰 수집 시작...
   🔄 4페이지 수집 중... (누적: 307개)
   ✨ 리뷰 완료: 307개

📍 2위 확정: [DOPAMY] 1172499524

   🚀 [【公式】日本初上陸！ ニューロペップ８R アンプルセラム 3...] 리뷰 수집 시작...
   🔄 8페이지 수집 중... (누적: 709개)
   ✨ 리뷰 완료: 709개

📍 3위 확정: [be bare] 1180516822

   🚀 [くるくる クレンジングバーム50ml...] 리뷰 수집 시작...
   🔄 3페이지 수집 중... (누적: 259개)
   ✨ 리뷰 완료: 259개

📍 4위 확정: [be bare] 1195669744

   🚀 [くるくるクレンジングバーム 50ml ＋ レフィル 50ml...] 리뷰 수집 시작...
   🔄 1페이지 수집 중... (누적: 5개)
   ✨ 리뷰 완료: 5개

📍 5위 확정: [ザツールラボ] 855174218

   🚀 [701 洗顔ブラシクレンジング ブラシ S...] 리뷰 수집 시작...
   🔄 1페이지 수집 중... (누적: 12개)
   ✨ 리뷰 완료: 12개

📍 6위 확정: [ザツールラボ] 855171577

   🚀 [1001 洗顔フォーム スウィッピング フェイス クレンザー...] 리뷰 수집 시작...
   🔄 1페이지 수집 중... (누적: 2개)
   ✨ 리뷰 완료: 2개

📍 7위 확정: [DOPAMY] 1175080093

   🚀 [【公式】日本初上陸！ 5点セット ニューロペップ８R　 クレ...] 리뷰 수집 시작...
   🔄 2페이지 수집 중... (누적: 157개)
   ✨ 리뷰 완료: 157개

📍 8위 확정: [DOPAMY] 1175081244

   🚀 [【公式】日本初上陸！ コンプリート6点セット ニューロペップ...

# 쿠텐 번역 코드

In [2]:
import json
import time
import re
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from deep_translator import GoogleTranslator

# ── 설정 ──────────────────────────────────────────────────────────
INPUT_FILE  = './qoo10_reviews_master.jsonl'
OUTPUT_FILE = 'qoo10_master_translated_jp_ko.jsonl' # 파일명에 en 추가
BODY_COL    = 'Review'    # 일본어 원문 키

# [병렬 처리 설정]
CHUNK_SIZE  = 5      # 묶음 번역 단위
MAX_WORKERS = 4      # 스레드 수 (구글 API 안정성을 위해 2 권장)
MAX_RETRIES = 3      # 실패 시 재시도
RETRY_SLEEP = 3.0    # 재시도 간격
CHUNK_DELAY = 1.0    # 번역 단계별 대기
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_numbered(texts: list) -> str:
    return "\n".join(f"[{i+1}] {str(t).strip()}" for i, t in enumerate(texts))

def parse_numbered(text: str, expected_n: int) -> list:
    pattern = re.compile(r'\[(\d+)\]\s*(.*?)(?=\[\d+\]|$)', re.DOTALL)
    found = pattern.findall(text)
    result = {int(idx): body.strip() for idx, body in found}
    return [result.get(i + 1, "") for i in range(expected_n)]

def translate_single(text: str, src: str, tgt: str) -> str:
    if not text or not str(text).strip(): return ""
    for attempt in range(MAX_RETRIES):
        try:
            res = GoogleTranslator(source=src, target=tgt).translate(text)
            if res: return res.strip()
        except:
            time.sleep(RETRY_SLEEP * (attempt + 1))
    return "번역실패"

def process_chunk(texts: list, src: str, tgt: str) -> list:
    if not texts: return []
    joined = build_numbered(texts)
    for attempt in range(MAX_RETRIES):
        try:
            translated = GoogleTranslator(source=src, target=tgt).translate(joined)
            if not translated: raise ValueError("응답 없음")
            parts = parse_numbered(translated, len(texts))
            if all(p.strip() for p in parts): return parts
            for i, p in enumerate(parts):
                if not p: parts[i] = translate_single(texts[i], src, tgt)
            return parts
        except:
            time.sleep(RETRY_SLEEP * (attempt + 1))
    return [translate_single(t, src, tgt) for t in texts]

def translate_workflow(texts: list):
    """ja -> en -> ko 2단계 번역 후 두 결과 모두 반환"""
    # 1단계: 일(ja) -> 영(en)
    en_texts = process_chunk(texts, 'ja', 'en')
    time.sleep(CHUNK_DELAY)
    
    # 2단계: 영(en) -> 한(ko)
    ko_texts = process_chunk(en_texts, 'en', 'ko')
    
    # 영어와 한국어 리스트를 튜플(tuple)로 반환하여 보존
    return en_texts, ko_texts

def main():
    print(f"📥 Qoo10 데이터 로딩 중: {INPUT_FILE}")
    records = []
    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip(): records.append(json.loads(line))
    except FileNotFoundError:
        print("❌ 입력 파일을 찾을 수 없습니다.")
        return

    df = pd.DataFrame(records)
    total_len = len(df)
    print(f"✅ 총 {total_len:,}건 확인 (일->영->한 번역 진행)")

    bodies = df[BODY_COL].fillna("").tolist()
    body_chunks = [bodies[i:i + CHUNK_SIZE] for i in range(0, total_len, CHUNK_SIZE)]
    
    all_body_en = [] # 영어 저장용 리스트
    all_body_ko = [] # 한국어 저장용 리스트
    start_time = time.time()
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # workflow에서 (영어리스트, 한국어리스트) 튜플 묶음이 반환됨
        results = list(tqdm(executor.map(translate_workflow, body_chunks), total=len(body_chunks), desc="번역 중"))
        
        for en_chunk, ko_chunk in results:
            all_body_en.extend(en_chunk)
            all_body_ko.extend(ko_chunk)
            
    # 데이터프레임에 두 컬럼 모두 추가
    df['Review_en'] = all_body_en[:total_len]
    df['Review_ko'] = all_body_ko[:total_len]
    elapsed = time.time() - start_time

    # JSONL 저장 (이제 Review_en 필드가 포함됨)
    print(f"\n💾 결과 저장 중: {OUTPUT_FILE}")
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        for record in df.to_dict(orient='records'):
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

    print(f"\n✨ 완료! 소요 시간: {elapsed/60:.1f}분")
    print(f"📊 처리 속도: {total_len/elapsed:.2f}건/초")

if __name__ == "__main__":
    main()

📥 Qoo10 데이터 로딩 중: ./qoo10_reviews_master.jsonl
✅ 총 2,834건 확인 (일->영->한 번역 진행)


번역 중: 100%|██████████| 567/567 [06:32<00:00,  1.44it/s]


💾 결과 저장 중: qoo10_master_translated_jp_ko.jsonl

✨ 완료! 소요 시간: 6.5분
📊 처리 속도: 7.22건/초


## 쿠텐 키워드 kebert 1차 분류

In [3]:
import re
import json
import pandas as pd
from tqdm import tqdm
from keybert import KeyBERT

# =============================================================================
# 1. 설정 및 경로
# =============================================================================
INPUT_FILE   = "./qoo10_master_translated_jp_ko.jsonl"
OUTPUT_CSV   = "./qoo10_keybert_en_categorized.csv"
OUTPUT_JSONL = "./qoo10_keybert_en_categorized.jsonl"

TEXT_COL       = "Review_en"
TOP_N_KEYWORDS = 5

CATEGORY_KEYWORDS = {
    "효과_성분": [
        "moistur", "hydrat", "hydrating", "whitening", "brighten", "elastic", "firm",
        "wrinkle", "anti-aging", "pore", "glow", "radian", "regenerat",
        "sooth", "calm", "antioxidant", "absorb", "penetrat", "tone", "improve",
        "vitamin", "retinol", "hyaluronic", "ceramide", "niacinamide", "peptid", "vegan",
        "plump", "clear", "even", "spot", "pigment", "aha", "bha", "acid", "exfoliat",
        # 추가: 피부타입 맥락
        "oily skin", "combination skin", "sensitive skin", "works for my skin",
        "dry skin type", "acne-prone skin",
        # 추가: 결과/시간 표현
        "result", "noticeabl", "overnight", "immediately", "after using",
        "after one week", "after a month", "after two week",
        # 추가: 사용 맥락
        "routine", "layer", "morning", "night cream",
        # 추가: 선케어 (제품군 해당 시)
        "spf", "sunscreen", "uv", "sun protect", "reef safe",
    ],
    "사용감_텍스처": [
        "appl", "blend", "texture", "consistenc", "watery", "runny", "thick", "viscos",
        "stick", "tacky", "fresh", "light", "weightless", "heavy", "soft", "smooth",
        "stiff", "greasy", "pill", "flake", "feel", "finish", "rub",
        "oily", "matte", "dewy", "sink", "patchy", "chalky", "white cast",
        # 추가: 메이크업 베이스/선케어 관련
        "pore-filling", "pore filling", "blur", "setting", "blot", "primer",
        "spread", "glide", "pack", "apply thin", "build up",
    ],
    "향_냄새": [
        "scent", "smell", "fragranc", "unscented", "fragrance-free", "odor",
        "subtle", "mild", "strong", "overpowering", "artificial", "natural", "perfume",
        "stink", "aroma", "nose",
        # 추가
        "whiff", "chemical smell", "medicin", "floral", "citrus",
    ],
    "피부_트러블_부작용": [
        "trouble", "breakout", "pimple", "acne", "irritat", "sting", "burn", "itch",
        "red", "redness", "peel", "tight", "sensitiv", "allerg", "dermatitis",
        "reaction", "side effect", "break out", "rash", "harsh",
        "drying", "dried out", "flaky", "dry patch",
        "clog", "purg", "cyst", "bump",
        # 추가: 자극 표현
        "tingle", "sting", "inflam", "swell", "hive", "welt",
        "made my skin worse", "broke me out", "not agree",
    ],
    "포장_배송": [
        "packag", "box", "bottle", "container", "case", "pump", "tube", "ship",
        "deliver", "late", "slow", "arriv", "damag", "broken",
        "leak", "spill", "wrap",
        "fast ship", "arrived fast", "quick deliver",
        "dropper", "cap", "lid", "spray", "nozzle", "shipped", "unseal",
        # 추가
        "packaging", "travel size", "full size", "well-packaged", "poorly packaged",
        "dent", "crush", "tamper",
    ],
    "가격_가성비": [
        "price", "cost", "valu", "expensiv", "pricy", "cheap", "afford", "reasonabl",
        "sale", "discount", "coupon", "buck", "money", "worth", "deal", "size", "amount",
        "pricey", "bargain", "rip off", "waste",
        # 추가: 용량 관련 가성비 표현
        "goes a long way", "a little goes", "last a long time", "last me",
        "small amount", "tiny bit", "lasts forever", "run out fast", "finish quickly",
    ],
    "고객서비스": [
        "custom", "service", "support", "respond", "response", "refund", "return",
        "exchang", "complain", "inquir", "answer", "contact", "issue",
        # 추가
        "seller", "vendor", "representative", "chat", "email them", "called",
        "waited", "resolve", "compensat",
    ],
    "제품불량": [
        "defect", "defective", "faulty", "bug", "dirt", "contaminat", "spoil",
        "weird", "fake", "counterfeit", "knockoff", "differ", "mold", "trash",
        "expir", "rancid", "separat", "empty",
        # 추가
        "smell off", "color off", "look different", "wrong product", "not what",
        "different from", "old stock", "bad batch",
    ],
    "재구매_추천": [
        "repurchas", "buy again", "reorder", "recommend", "holy grail", "staple",
        "go-to", "favorit", "gift", "friend", "keep us", "definitely",
        "love it",
        "hg", "restock", "10/10", "must have", "obsessed",
        # 추가: 비교/전환 표현
        "better than", "switch from", "switch to", "compared to", "used to use",
        "replace", "converted",
        # 추가: 일반 강한 긍정 (내용 없는 짧은 리뷰 흡수)
        "highly recommend", "great product", "works well", "works great",
        "amazing product", "excellent", "perfect product", "love this",
        "love how", "love that", "so good", "so happy",
    ],
    "부정_리뷰": [
        # 추가: 부정 추천 표현
        "disappoint", "not recommend", "waste of money", "regret buying",
    ],

    "커버력_색상": [
        "cover", "coverage", "color", "shade", "tone", "tint", "pigment",
        "bright", "dark", "ashy", "oxidiz", "orang", "yellow", "match", "pale",
        "undertone", "fair", "sheer", "opaque", "swatch",
        # 추가
        "full coverage", "medium coverage", "buildable", "natural finish",
        "foundation", "concealer", "bb cream", "cc cream", "tinted",
        "skin tone", "complexion", "too light", "too dark", "perfect match",
    ],
    "지속력_밀착력": [
        "last", "lasting", "longevity", "stay", "adher", "crease", "melt", "fade",
        "long-lasting", "all day", "wear", "hold", "slip", "smudg", "transfer",
        "rub off", "budge", "separate",
        # 추가: 환경 내구성
        "sweat", "sweatproof", "waterproof", "water resistant", "humid",
        "through the day", "by noon", "by midday", "hours later",
        "8 hour", "12 hour", "24 hour",
    ],
}

# =============================================================================
# 2. 유틸리티 함수
# =============================================================================
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

def clean_light(text):
    text = str(text).replace("\n", " ").replace("\r", " ")
    return re.sub(r"\s+", " ", text).strip()

def extract_keywords_func(text, model, top_n=5):
    text = str(text).strip()
    if not text:
        return []
    try:
        kws = model.extract_keywords(
            text,
            keyphrase_ngram_range=(1, 2),
            stop_words="english",
            top_n=top_n,
            use_mmr=True,
            diversity=0.5
        )
        return [kw for kw, _ in kws]
    except Exception:
        return []

def rule_classify(keywords: list, text: str) -> tuple:
    combined = " ".join(keywords).lower() + " " + text.lower()
    scores = {}
    for cat, kw_list in CATEGORY_KEYWORDS.items():
        count = sum(1 for kw in kw_list if kw in combined)
        if count > 0:
            scores[cat] = count

    if not scores:
        return "unclassified", ["unclassified"]

    # 점수 높은 순 정렬
    sorted_cats = sorted(scores, key=scores.get, reverse=True)
    return sorted_cats[0], sorted_cats[:3]

# =============================================================================
# 3. 메인 실행 프로세스
# =============================================================================
if __name__ == "__main__":
    # 모델 로드
    print("🚀 KeyBERT 모델 로딩 중...")
    kw_model = KeyBERT("all-MiniLM-L6-v2")

    # 데이터 로드 및 전처리
    df = load_jsonl(INPUT_FILE)
    df = df.dropna(subset=[TEXT_COL]).copy()
    df = df[df[TEXT_COL].astype(str).str.strip() != ""].reset_index(drop=True)
    df["text_for_model"] = df[TEXT_COL].apply(clean_light)
    
    print(f"📊 유효 데이터: {len(df):,}건")

    # 결과 저장을 위한 리스트
    keywords_list      = []
    primary_categories = []
    categories_list    = []

    # 키워드 추출 및 분류 루프
    for text in tqdm(df["text_for_model"], desc="KeyBERT 분류 (EN)"):
        kws = extract_keywords_func(text, kw_model, top_n=TOP_N_KEYWORDS)
        primary, cats = rule_classify(kws, text)
        
        keywords_list.append(kws)
        primary_categories.append(primary)
        categories_list.append(cats)

    # 데이터프레임 할당
    df["keybert_keywords"] = keywords_list
    df["primary_category"] = primary_categories
    df["categories"]       = categories_list

    # 통계 출력
    total        = len(df)
    classified   = (df["primary_category"] != "unclassified").sum()
    unclassified = (df["primary_category"] == "unclassified").sum()

    print(f"\n✅ 분류 완료: {classified:,}건 ({classified/total*100:.1f}%)")
    print(f"❌ 미분류  : {unclassified:,}건 ({unclassified/total*100:.1f}%)")
    print("\n[카테고리 분포]")
    print(df["primary_category"].value_counts())

    # 저장
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\n")

    print(f"\n💾 저장 완료:")
    print(f" - CSV  : {OUTPUT_CSV}")
    print(f" - JSONL: {OUTPUT_JSONL}")

    print("\n🔍 상위 10개 샘플:")
    print(df[[TEXT_COL, "keybert_keywords", "primary_category"]].head(10))

🚀 KeyBERT 모델 로딩 중...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📊 유효 데이터: 2,834건


KeyBERT 분류 (EN): 100%|██████████| 2834/2834 [00:59<00:00, 47.43it/s] 



✅ 분류 완료: 2,397건 (84.6%)
❌ 미분류  : 437건 (15.4%)

[카테고리 분포]
primary_category
효과_성분           937
사용감_텍스처         900
unclassified    437
포장_배송           177
가격_가성비          142
피부_트러블_부작용       89
재구매_추천           67
향_냄새             55
지속력_밀착력          15
커버력_색상           11
고객서비스             3
제품불량              1
Name: count, dtype: int64

💾 저장 완료:
 - CSV  : ./qoo10_keybert_en_categorized.csv
 - JSONL: ./qoo10_keybert_en_categorized.jsonl

🔍 상위 10개 샘플:
                                           Review_en  \
0  I won at the sample market 🔥 [Neuropep 8 Reset...   
1  I was interested in this because it uses the m...   
2  I saw the phrase "first landing in Japan" in s...   
3  I won at the sample market! Thank you 😊 The cl...   
4  I received the lotion and serum at the sample ...   
5  Image ↓↓↓Right cheek before use (at night) Red...   
6  I won at the sample market! thank you! The ser...   
7  I won the sample market this time and was able...   
8  I won the toner and serum that I wan

## 쿠텐 키워드 gpt 2차분류

In [4]:
import json
import re
from openai import OpenAI

# ── 설정 ──────────────────────────────────────────────────────────────
INPUT_JSONL  = "./qoo10_keybert_en_categorized.jsonl"  # KeyBERT 결과 JSONL
OUTPUT_JSONL = "./qoo10_final_categorized.jsonl"
BATCH_SIZE   = 30
TEXT_COL     = "Review_en"

CATEGORIES = [
    "효과_성분", "사용감_텍스처", "향_냄새", "피부_트러블_부작용","부정_리뷰",
    "포장_배송", "가격_가성비", "고객서비스", "제품불량",
    "재구매_추천", "커버력_색상", "지속력_밀착력", "미분류"
]

client = OpenAI()

# ── GPT 배치 분류 ──────────────────────────────────────────────────────
def gpt_classify_batch(batch: list[dict]) -> dict:
    """batch: [{"idx": i, "keywords": [...], "text": "..."}]
    반환: {idx: {"primary_category": ..., "categories": [...]}}
    """
    prompt_items = "\n".join(
        f"[{item['idx']}] keywords={item['keywords']} | text={item['text'][:300]}"
        for item in batch
    )
    system_msg = f"""뷰티 제품 리뷰를 아래 카테고리 중 하나로 분류하세요.
카테고리: {CATEGORIES}

각 리뷰에 대해 JSON 배열로 응답하세요:
[{{"idx": 번호, "primary_category": "카테고리명", "categories": ["카테고리1", ...]}}]
primary_category는 가장 핵심 카테고리 1개, categories는 해당되는 카테고리 모두."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": prompt_items}
        ],
        temperature=0
    )

    raw = response.choices[0].message.content
    # JSON 배열 파싱
    match = re.search(r'\[.*\]', raw, re.DOTALL)
    if not match:
        return {}
    results = json.loads(match.group())
    return {r["idx"]: r for r in results}

# ── 데이터 로드 ────────────────────────────────────────────────────────
records = []
with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

classified   = [r for r in records if r.get("primary_category") != "unclassified"]
unclassified = [r for r in records if r.get("primary_category") == "unclassified"]

print(f"전체: {len(records)} | 분류완료: {len(classified)} | GPT 재분류 대상: {len(unclassified)}")

# ── GPT 배치 실행 ──────────────────────────────────────────────────────
idx_to_record = {i: rec for i, rec in enumerate(unclassified)}
batches = [
    [
        {
            "idx": i,
            "keywords": rec.get("keybert_keywords", []),
            "text": str(rec.get(TEXT_COL, ""))[:300]
        }
        for i, rec in list(idx_to_record.items())[start:start+BATCH_SIZE]
    ]
    for start in range(0, len(unclassified), BATCH_SIZE)
]

print(f"배치 수: {len(batches)} ({BATCH_SIZE}건씩)")

for b_idx, batch in enumerate(batches):
    results = gpt_classify_batch(batch)
    for item in batch:
        i = item["idx"]
        if i in results:
            idx_to_record[i]["primary_category"] = results[i]["primary_category"]
            idx_to_record[i]["categories"]       = results[i]["categories"]
        else:
            idx_to_record[i]["primary_category"] = "이분류"
            idx_to_record[i]["categories"]       = []
    print(f"  배치 {b_idx+1}/{len(batches)} 완료")

# ── 결과 저장 ──────────────────────────────────────────────────────────
final_records = classified + list(idx_to_record.values())

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for rec in final_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"\n저장 완료: {OUTPUT_JSONL} ({len(final_records)}건)")

# ── 분류 결과 확인 ─────────────────────────────────────────────────────
from collections import Counter
cats = [r["primary_category"] for r in final_records]
for cat, cnt in Counter(cats).most_common():
    print(f"  {cat}: {cnt}")

전체: 2834 | 분류완료: 2397 | GPT 재분류 대상: 437
배치 수: 15 (30건씩)
  배치 1/15 완료
  배치 2/15 완료
  배치 3/15 완료
  배치 4/15 완료
  배치 5/15 완료
  배치 6/15 완료
  배치 7/15 완료
  배치 8/15 완료
  배치 9/15 완료
  배치 10/15 완료
  배치 11/15 완료
  배치 12/15 완료
  배치 13/15 완료
  배치 14/15 완료
  배치 15/15 완료

저장 완료: ./qoo10_final_categorized.jsonl (2834건)
  효과_성분: 1015
  사용감_텍스처: 940
  미분류: 250
  포장_배송: 178
  가격_가성비: 144
  재구매_추천: 122
  피부_트러블_부작용: 92
  향_냄새: 55
  지속력_밀착력: 16
  커버력_색상: 11
  부정_리뷰: 7
  고객서비스: 3
  제품불량: 1


## 효과_성분 세분화 KeyBERT 재분류 + GPT 2차 분류

In [3]:
import re
import json
import pandas as pd
from tqdm import tqdm
from keybert import KeyBERT
from openai import OpenAI
from collections import Counter

# =============================================================================
# 설정
# =============================================================================
INPUT_FILE   = "./qoo10_master_translated_jp_ko.jsonl"
OUTPUT_JSONL = "./qoo10_final_categorized_v2.jsonl"
TEXT_COL     = "Review_en"
BATCH_SIZE   = 30

CATEGORY_KEYWORDS = {
    "주름_노화": [
        "wrinkle", "anti-aging", "antiaging", "fine line", "age spot", "sagging",
        "retinol", "peptid", "adenosine", "bakuchiol", "firm", "elastic", "lifting",
        "tighten", "regenerat",
    ],
    "보습_수분": [
        "moistur", "hydrat", "hydrating", "plump", "dehydrat", "quench", "dewiness",
        "hyaluronic", "ceramide", "glycerin", "water",
    ],
    "미백_브라이트닝": [
        "whitening", "brighten", "bright", "glow", "radian", "tone", "even", "clear",
        "spot", "pigment", "discolor", "dull",
        "niacinamide", "vitamin c", "arbutin", "tranexamic",
    ],
    "진정_장벽": [
        "sooth", "calm", "barrier", "sensitiv",
        "centella", "cica", "madecassoside", "panthenol", "allantoin", "madeca",
    ],
    "모공_각질": [
        "pore", "exfoliat", "aha", "bha", "acid", "peel", "blackhead",
        "sebum", "rough", "bumpy",
    ],
    "사용감_텍스처": [
        "appl", "blend", "texture", "consistenc", "watery", "runny", "thick", "viscos",
        "stick", "tacky", "fresh", "light", "weightless", "heavy", "soft", "smooth",
        "stiff", "greasy", "pill", "flake", "feel", "finish", "rub",
        "oily", "matte", "dewy", "sink", "patchy", "chalky", "white cast",
        "pore-filling", "blur", "setting", "blot", "primer", "spread", "glide",
    ],
    "향_냄새": [
        "scent", "smell", "fragranc", "unscented", "fragrance-free", "odor",
        "subtle", "mild", "strong", "overpowering", "artificial", "natural", "perfume",
        "stink", "aroma", "nose", "whiff", "chemical smell", "floral", "citrus",
    ],
    "피부_트러블_부작용": [
        "trouble", "breakout", "pimple", "acne", "irritat", "sting", "burn", "itch",
        "red", "redness", "tight", "allerg", "dermatitis", "reaction", "side effect",
        "break out", "rash", "harsh", "drying", "dried out", "flaky", "dry patch",
        "clog", "purg", "cyst", "bump", "tingle", "inflam", "swell",
        "made my skin worse", "broke me out",
    ],
    "포장_배송": [
        "packag", "box", "bottle", "container", "case", "pump", "tube", "ship",
        "deliver", "late", "slow", "arriv", "damag", "broken", "leak", "spill",
        "dropper", "cap", "lid", "spray", "nozzle", "dent", "crush",
    ],
    "가격_가성비": [
        "price", "cost", "valu", "expensiv", "pricy", "cheap", "afford", "reasonabl",
        "sale", "discount", "money", "worth", "deal", "pricey", "bargain",
        "goes a long way", "a little goes", "lasts forever",
    ],
    "재구매_추천": [
        "repurchas", "buy again", "reorder", "recommend", "holy grail", "staple",
        "favorit", "must have", "obsessed", "better than", "switch from",
        "highly recommend", "great product", "works well", "love this", "so good",
    ],
    "부정_리뷰": [
        "disappoint", "not recommend", "waste of money", "regret buying",
    ],
    "지속력_밀착력": [
        "last", "lasting", "longevity", "stay", "adher", "crease", "melt", "fade",
        "long-lasting", "all day", "wear", "hold", "smudg", "transfer", "rub off",
        "sweatproof", "waterproof", "water resistant",
    ],
    "고객서비스": [
        "custom", "service", "support", "respond", "refund", "return", "exchang",
        "complain", "answer", "contact", "seller", "vendor",
    ],
    "제품불량": [
        "defect", "defective", "faulty", "contaminat", "spoil", "fake", "counterfeit",
        "expir", "rancid", "wrong product", "different from", "bad batch",
    ],
}

CATEGORIES = list(CATEGORY_KEYWORDS.keys()) + ["미분류"]

# =============================================================================
# Step 1. KeyBERT 1차 분류
# =============================================================================
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

def clean_light(text):
    text = str(text).replace("\n", " ").replace("\r", " ")
    return re.sub(r"\s+", " ", text).strip()

def extract_keywords_func(text, model, top_n=5):
    text = str(text).strip()
    if not text:
        return []
    try:
        kws = model.extract_keywords(
            text, keyphrase_ngram_range=(1, 2),
            stop_words="english", top_n=top_n,
            use_mmr=True, diversity=0.5
        )
        return [kw for kw, _ in kws]
    except Exception:
        return []

def rule_classify(keywords: list, text: str) -> tuple:
    combined = " ".join(keywords).lower() + " " + text.lower()
    scores = {}
    for cat, kw_list in CATEGORY_KEYWORDS.items():
        count = sum(1 for kw in kw_list if kw in combined)
        if count > 0:
            scores[cat] = count
    if not scores:
        return "unclassified", ["unclassified"]
    sorted_cats = sorted(scores, key=scores.get, reverse=True)
    return sorted_cats[0], sorted_cats[:3]

print("🚀 KeyBERT 모델 로딩 중...")
kw_model = KeyBERT("all-MiniLM-L6-v2")

df = load_jsonl(INPUT_FILE)
df = df.dropna(subset=[TEXT_COL]).copy()
df = df[df[TEXT_COL].astype(str).str.strip() != ""].reset_index(drop=True)
df["text_for_model"] = df[TEXT_COL].apply(clean_light)
print(f"📊 유효 데이터: {len(df):,}건")

keywords_list, primary_categories, categories_list = [], [], []
for text in tqdm(df["text_for_model"], desc="KeyBERT 1차 분류"):
    kws = extract_keywords_func(text, kw_model)
    primary, cats = rule_classify(kws, text)
    keywords_list.append(kws)
    primary_categories.append(primary)
    categories_list.append(cats)

df["keybert_keywords"] = keywords_list
df["primary_category"] = primary_categories
df["categories"]       = categories_list

classified_df   = df[df["primary_category"] != "unclassified"]
unclassified_df = df[df["primary_category"] == "unclassified"]
print(f"\n✅ 1차 분류 완료: {len(classified_df):,}건 | GPT 대상: {len(unclassified_df):,}건")
print(classified_df["primary_category"].value_counts())

# =============================================================================
# Step 2. GPT 2차 분류 (unclassified만)
# =============================================================================
client = OpenAI()

def gpt_classify_batch(batch: list) -> dict:
    prompt_items = "\n".join(
        f"[{item['idx']}] keywords={item['keywords']} | text={item['text'][:300]}"
        for item in batch
    )
    system_msg = f"""뷰티 제품 리뷰를 아래 카테고리 중 하나로 분류하세요.
카테고리: {CATEGORIES}

각 리뷰에 대해 JSON 배열로 응답하세요:
[{{"idx": 번호, "primary_category": "카테고리명", "categories": ["카테고리1", ...]}}]
primary_category는 가장 핵심 카테고리 1개, categories는 해당되는 카테고리 모두."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user",   "content": prompt_items},
        ],
        temperature=0,
    )
    raw = response.choices[0].message.content
    match = re.search(r'\[.*\]', raw, re.DOTALL)
    if not match:
        return {}
    results = json.loads(match.group())
    return {r["idx"]: r for r in results}

records              = df.to_dict("records")
classified_records   = [r for r in records if r["primary_category"] != "unclassified"]
unclassified_records = [r for r in records if r["primary_category"] == "unclassified"]

idx_to_record = {i: rec for i, rec in enumerate(unclassified_records)}
batches = [
    [
        {
            "idx":      i,
            "keywords": rec.get("keybert_keywords", []),
            "text":     str(rec.get(TEXT_COL, ""))[:300],
        }
        for i, rec in list(idx_to_record.items())[s:s + BATCH_SIZE]
    ]
    for s in range(0, len(unclassified_records), BATCH_SIZE)
]

print(f"\n🤖 GPT 2차 분류 시작: {len(batches)}배치")
for b_idx, batch in enumerate(batches):
    results = gpt_classify_batch(batch)
    for item in batch:
        i = item["idx"]
        if i in results:
            idx_to_record[i]["primary_category"] = results[i]["primary_category"]
            idx_to_record[i]["categories"]       = results[i]["categories"]
        else:
            idx_to_record[i]["primary_category"] = "미분류"
            idx_to_record[i]["categories"]       = []
    print(f"  배치 {b_idx+1}/{len(batches)} 완료")

# =============================================================================
# 저장
# =============================================================================
final_records = classified_records + list(idx_to_record.values())

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for rec in final_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"\n💾 저장 완료: {OUTPUT_JSONL} ({len(final_records):,}건)")
print("\n[최종 카테고리 분포]")
for cat, cnt in Counter(r["primary_category"] for r in final_records).most_common():
    print(f"  {cat}: {cnt:,}")


🚀 KeyBERT 모델 로딩 중...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📊 유효 데이터: 2,834건


KeyBERT 1차 분류: 100%|██████████| 2834/2834 [05:46<00:00,  8.18it/s]



✅ 1차 분류 완료: 2,249건 | GPT 대상: 585건
primary_category
사용감_텍스처       1022
미백_브라이트닝       280
포장_배송          198
보습_수분          176
가격_가성비         152
향_냄새            87
피부_트러블_부작용      86
모공_각질           81
주름_노화           70
재구매_추천          44
진정_장벽           26
지속력_밀착력         19
고객서비스            5
제품불량             2
부정_리뷰            1
Name: count, dtype: int64

🤖 GPT 2차 분류 시작: 20배치
  배치 1/20 완료
  배치 2/20 완료
  배치 3/20 완료
  배치 4/20 완료
  배치 5/20 완료
  배치 6/20 완료
  배치 7/20 완료
  배치 8/20 완료
  배치 9/20 완료
  배치 10/20 완료
  배치 11/20 완료
  배치 12/20 완료
  배치 13/20 완료
  배치 14/20 완료
  배치 15/20 완료
  배치 16/20 완료
  배치 17/20 완료
  배치 18/20 완료
  배치 19/20 완료
  배치 20/20 완료

💾 저장 완료: ./qoo10_final_categorized_v2.jsonl (2,834건)

[최종 카테고리 분포]
  사용감_텍스처: 1,121
  미분류: 354
  미백_브라이트닝: 290
  보습_수분: 210
  포장_배송: 203
  가격_가성비: 154
  재구매_추천: 111
  피부_트러블_부작용: 87
  향_냄새: 87
  모공_각질: 82
  주름_노화: 76
  진정_장벽: 27
  지속력_밀착력: 19
  고객서비스: 5
  부정_리뷰: 5
  제품불량: 2
  효과_효능: 1


In [4]:
import json
from collections import Counter

V2_TO_V1 = {
    "보습_수분":       "효과_성분",
    "미백_브라이트닝": "효과_성분",
    "모공_각질":       "효과_성분",
    "주름_노화":       "효과_성분",
    "진정_장벽":       "효과_성분",
}

records = []
with open("qoo10_final_categorized_v2.jsonl", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        obj = json.loads(line)
        obj["primary_category_v2"] = obj["primary_category"]
        obj["categories_v2"] = obj.get("categories", [])
        obj["primary_category"] = V2_TO_V1.get(obj["primary_category"], obj["primary_category"])
        seen = []
        for cat in obj.get("categories", []):
            mapped = V2_TO_V1.get(cat, cat)
            if mapped not in seen:
                seen.append(mapped)
        obj["categories"] = seen
        records.append(obj)

# 분포 확인
c1 = Counter(r["primary_category"]    for r in records)
c2 = Counter(r["primary_category_v2"] for r in records)
print("=== v1 (대시보드) ===")
for k, v in sorted(c1.items()): print(f"  {k}: {v}")
print("\n=== v2 (점수산출) ===")
for k, v in sorted(c2.items()): print(f"  {k}: {v}")

# 샘플 확인
s = records[0]
print(f"\nprimary_category   : {s['primary_category']}")
print(f"categories         : {s['categories']}")
print(f"primary_category_v2: {s['primary_category_v2']}")
print(f"categories_v2      : {s['categories_v2']}")

# 저장
with open("qoo10_final_categorized_v2.jsonl", "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"\n저장 완료: {len(records)}개")

=== v1 (대시보드) ===
  가격_가성비: 154
  고객서비스: 5
  미분류: 354
  부정_리뷰: 5
  사용감_텍스처: 1121
  재구매_추천: 111
  제품불량: 2
  지속력_밀착력: 19
  포장_배송: 203
  피부_트러블_부작용: 87
  향_냄새: 87
  효과_성분: 685
  효과_효능: 1

=== v2 (점수산출) ===
  가격_가성비: 154
  고객서비스: 5
  모공_각질: 82
  미백_브라이트닝: 290
  미분류: 354
  보습_수분: 210
  부정_리뷰: 5
  사용감_텍스처: 1121
  재구매_추천: 111
  제품불량: 2
  주름_노화: 76
  지속력_밀착력: 19
  진정_장벽: 27
  포장_배송: 203
  피부_트러블_부작용: 87
  향_냄새: 87
  효과_효능: 1

primary_category   : 사용감_텍스처
categories         : ['사용감_텍스처', '피부_트러블_부작용', '효과_성분']
primary_category_v2: 사용감_텍스처
categories_v2      : ['사용감_텍스처', '피부_트러블_부작용', '보습_수분']

저장 완료: 2834개
